# Airline Reviews Pipeline Updated(1)

Primary six-source ingestion, cleaning and translation. Use canonical combined exports only. Set TRANSLATE_REVIEWS=False for a dry run; existing outputs can be overwritten.

**Execution:** Start Jupyter inside the project tree. Run the path setup first. Review configuration and existing outputs before executing.

In [ ]:
# Resolve the project from the notebook working directory, not a machine-specific path.
from pathlib import Path
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if all((p / n).is_dir() for n in ("Python Code", "Data", "Streamlit"))), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Launch Jupyter from inside the AEGEAN project tree.")
SCRAPING_DIR = PROJECT_ROOT / "Data" / "Scraping"
CLEANING_DIR = PROJECT_ROOT / "Data" / "Output" / "Cleaning data"
HF_DIR = PROJECT_ROOT / "Data" / "Output" / "Hugging Face"
LLM_OUTPUT_DIR = PROJECT_ROOT / "Data" / "Output" / "LLM"
for directory in (CLEANING_DIR, HF_DIR, LLM_OUTPUT_DIR):
    directory.mkdir(parents=True, exist_ok=True)


# Airline Review Data Pipeline — Six Sources

**Sources:** AirlineRatings, Flight-Report, eSky, Kayak, TripAdvisor, Trustpilot.

This notebook consolidates source CSV files, validates and cleans records, detects original languages, translates non-English reviews, and exports datasets and quality-control reports. It preserves the original review and source rating; ratings are **not automatically converted to a common scale**.

**Important:** The uploaded notebook documented the first four source schemas only. TripAdvisor and Trustpilot field aliases below are configurable hypotheses, not verified schemas. Inspect the ingestion report before translation. Missing optional source files are reported; required files are configurable.


## Verified source schemas (11 November 2026 samples)
TripAdvisor: `locationName` identifies the airline (e.g. AEGEAN); `source_review_id`, `review_date`, `rating`, `review_title`, `review`, and `language` are available.
Trustpilot: `airline`, `source_review_id`, `review_date`, `rating`, `review_title`, `review`, and `language` are available.
The supplied Excel samples each contain CSV-formatted text in a single worksheet column, not separate Excel fields. The actual source files are read as CSV, as intended.


In [ ]:
# Run once in your Jupyter environment if needed:
# %pip install -U pandas numpy openpyxl lingua-language-detector tqdm openai airportsdata


In [ ]:
from __future__ import annotations

import hashlib
import json
import logging
import os
import re
import time
from dataclasses import dataclass
from getpass import getpass
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger("airline_reviews")

# Original notebook: current notebook directory -> parent -> Data
DATA_DIR = SCRAPING_DIR
OUTPUT_DIR = CLEANING_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRANSLATE_REVIEWS = True  # Set False to validate the dataset without API calls
TRANSLATION_MODEL = "gpt-4.1-mini"
SAVE_EVERY = 25
MAX_TRANSLATION_ATTEMPTS = 4
SLEEP_BETWEEN_CALLS = 0.1

# CSV files may contain review content with commas and line breaks.
CSV_ENCODING = "utf-8-sig"

# Configure filenames/patterns to match the actual files in Data/.
# Multiple matching files for a source are combined, so do not keep overlapping exports.
SOURCE_PATTERNS = {
    "AirlineRatings": ["airlineratings_data/all_airlines_reviews.csv"],
    "Flight-Report": ["flight_report_data/all_airlines_reviews.csv"],
    "eSky": ["esky/esky_all_reviews_v2.csv"],
    "Kayak": ["kayak/kayak_all_airline_reviews.csv"],
    "TripAdvisor": ["tripadvisor_reviews/tripadvisor_all_airlines.csv"],
    "Trustpilot": ["trustpilot_reviews/trustpilot_all_airlines.csv"],
}
REQUIRED_SOURCES = {"AirlineRatings", "Flight-Report", "eSky", "Kayak"}

print("Data folder:", DATA_DIR.resolve())
print("Output folder:", OUTPUT_DIR.resolve())
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Data folder not found: {DATA_DIR.resolve()}")


In [ ]:
# Source-specific mappings are checked FIRST, followed by generic aliases.
# Adjust these if the TripAdvisor/Trustpilot exports use different headings.
SOURCE_FIELDS = {
    "AirlineRatings": {
        "review_date": ["review_date"], "rating": ["rating_listing"],
        "review": ["excerpt_listing", "review_text", "review"],
    },
    "Flight-Report": {
        "review_date": ["publication_date"], "rating": ["airline_overall"],
        "review": ["conclusion"],
    },
    "eSky": {
        "review_date": ["publication_date"], "rating": ["overall_rating"],
        "review": ["review_text"],
    },
    "Kayak": {
        "review_date": ["publication_date"], "rating": ["overall_rating"],
        "review": ["review_text"],
    },
    "TripAdvisor": {
        "review_date": ["review_date", "createdDate", "publishedAt", "publication_date", "date", "published_date"],
        "rating": ["overall_rating", "rating", "bubble_rating", "score"],
        "review": ["review_text", "review", "text", "content", "body"],
    },
    "Trustpilot": {
        "review_date": ["review_date", "publishedAt", "createdAt", "dateOfExperience", "date"],
        "rating": ["stars", "rating", "score"],
        "review": ["review", "text", "review_text", "content", "body"],
    },
}

COMMON_FIELDS = {
    "airline": ["airline", "airline_name", "locationName", "company", "company_name", "airline_slug", "brand"],
    "review_date": ["review_date", "publication_date", "published_at", "publishedAt", "created_at", "createdAt", "date", "datePublished"],
    "rating": ["rating", "overall_rating", "rating_listing", "airline_overall", "stars", "score"],
    "review": ["review", "review_text", "excerpt_listing", "conclusion", "text", "content", "body", "description"],
    "review_id": ["source_review_id", "review_id", "id", "reviewId", "review_url", "url"],
    "title": ["review_title", "title", "headline"],
    "origin": ["origin", "origin_airport", "departure_airport", "from"],
    "destination": ["destination", "destination_airport", "arrival_airport", "to"],
    "route": ["route", "flight_route"],
    "flight_number": ["flight_number", "flight_no", "flightNumber"],
}

AIRLINE_NAMES = {
    "aegean_airlines": "Aegean Airlines", "aegean": "Aegean Airlines",
    "air_france": "Air France", "air france": "Air France",
    "bangkok_airways": "Bangkok Air", "bangkok airways": "Bangkok Air",
    "bangkok air": "Bangkok Air", "lufthansa": "Lufthansa",
    "porter_airlines": "Porter Airlines", "porter airlines": "Porter Airlines",
    "turkish_airlines": "Turkish Airlines", "turkish airlines": "Turkish Airlines",
    "virgin_australia": "Virgin Australia", "virgin australia": "Virgin Australia",
}

# Used ONLY if an export lacks an airline column and its filename clearly identifies one.
AIRLINE_FILENAME_ALIASES = {
    "aegean": "Aegean Airlines", "turkish": "Turkish Airlines",
    "air_france": "Air France", "air-france": "Air France",
    "lufthansa": "Lufthansa", "bangkok": "Bangkok Air",
    "porter": "Porter Airlines", "virgin": "Virgin Australia",
}

STANDARD_COLUMNS = ["airline", "review_date", "rating", "review", "source",
                    "review_id", "title", "origin", "destination", "route", "flight_number"]

def normalize_column(name):
    """Return or process normalize column."""
    return re.sub(r"[^a-z0-9]", "", str(name).casefold())

def find_column(columns, candidates):
    """Return or process find column."""
    lookup = {normalize_column(col): col for col in columns}
    for candidate in candidates:
        actual = lookup.get(normalize_column(candidate))
        if actual is not None:
            return actual
    return None

def airline_from_filename(path):
    """Return or process airline from filename."""
    stem = path.stem.casefold().replace("-", "_").replace(" ", "_")
    matches = {canonical for alias, canonical in AIRLINE_FILENAME_ALIASES.items()
               if alias.replace("-", "_").replace(" ", "_") in stem}
    return next(iter(matches)) if len(matches) == 1 else None

def clean_text(series):
    """Return or process clean text."""
    s = series.astype("string").str.replace(r"\s+", " ", regex=True).str.strip()
    return s.mask(s.str.casefold().isin(["", "nan", "none", "null", "nat"]))

def parse_dates(series, source):
    """Return or process parse dates."""
    s = clean_text(series)
    if source == "AirlineRatings":
        primary = pd.to_datetime(s, format="%d %B, %Y", errors="coerce", utc=True)
    elif source == "Kayak":
        primary = pd.to_datetime(s, format="%Y-%m", errors="coerce", utc=True)
    else:
        primary = pd.to_datetime(s, format="mixed", errors="coerce", utc=True)
    missing = primary.isna() & s.notna()
    if missing.any():
        primary.loc[missing] = pd.to_datetime(s.loc[missing], errors="coerce", utc=True, dayfirst=True)
    return primary.dt.tz_convert(None).dt.normalize()

def parse_ratings(series):
    """Return or process parse ratings."""
    s = clean_text(series).str.replace(",", ".", regex=False)
    # Only parse a plain numeric value or an explicit x/y rating.
    values = s.str.extract(r"^([0-9]+(?:\.[0-9]+)?)(?:\s*/\s*[0-9]+(?:\.[0-9]+)?)?$", expand=False)
    return pd.to_numeric(values, errors="coerce")

def normalize_airline(value):
    """Return or process normalize airline."""
    if pd.isna(value):
        return pd.NA
    name = str(value).strip()
    key = re.sub(r"\s+", " ", name.replace("_", " ")).casefold()
    return AIRLINE_NAMES.get(key, name)


In [ ]:
def read_source_csv(path):
    """Return or process read source csv."""
    try:
        return pd.read_csv(path, encoding=CSV_ENCODING, low_memory=False, dtype="string")
    except UnicodeDecodeError:
        logger.warning("Using cp1252 fallback: %s", path.name)
        return pd.read_csv(path, encoding="cp1252", low_memory=False, dtype="string")

def standardize_file(raw, source, path):
    """Return or process standardize file."""
    specific = SOURCE_FIELDS.get(source, {})
    mapped = {}
    for field in STANDARD_COLUMNS:
        candidates = specific.get(field, []) + COMMON_FIELDS.get(field, [])
        selected = find_column(raw.columns, candidates)
        mapped[field] = raw[selected] if selected is not None else pd.Series(pd.NA, index=raw.index, dtype="string")

    frame = pd.DataFrame(mapped)
    if frame["airline"].isna().all():
        fallback = airline_from_filename(path)
        if fallback:
            frame["airline"] = fallback
            logger.info("Airline inferred from filename: %s -> %s", path.name, fallback)

    # TripAdvisor export uses locationName for airline; Trustpilot uses airline.
    # Both provide source_review_id, review_date, rating, review, review_title.
    # Fail loudly on an unrecognized review schema; do not lose data silently.
    if frame["review"].isna().all() and len(raw):
        raise ValueError(f"No recognized review text column in {path.name}. Columns: {list(raw.columns)}")
    if frame["airline"].isna().all() and len(raw):
        raise ValueError(f"No airline column or unambiguous airline filename in {path.name}. Columns: {list(raw.columns)}")

    frame["source"] = source
    frame["source_file"] = path.name
    frame["airline"] = frame["airline"].map(normalize_airline).astype("string")
    for col in ["review", "review_id", "title", "origin", "destination", "route", "flight_number"]:
        frame[col] = clean_text(frame[col])
    frame["review_date"] = parse_dates(frame["review_date"], source)
    frame["rating"] = parse_ratings(frame["rating"])
    frame["rating_original_scale"] = pd.NA  # Never assume comparable scales.
    return frame

frames, ingest_log = [], []
for source, patterns in SOURCE_PATTERNS.items():
    paths = sorted({p for pattern in patterns for p in DATA_DIR.glob(pattern) if p.is_file()})
    if not paths:
        message = f"No files found for {source}: {patterns}"
        if source in REQUIRED_SOURCES:
            raise FileNotFoundError(message)
        logger.warning(message)
        ingest_log.append({"source": source, "file": "", "rows": 0, "status": "not_found"})
        continue
    for path in paths:
        raw = read_source_csv(path)
        standardized = standardize_file(raw, source, path)
        frames.append(standardized)
        ingest_log.append({"source": source, "file": path.name, "rows": len(raw), "status": "loaded",
                           "missing_review": int(standardized["review"].isna().sum()),
                           "missing_date": int(standardized["review_date"].isna().sum()),
                           "missing_rating": int(standardized["rating"].isna().sum()),
                           "columns": ", ".join(raw.columns)})
        logger.info("Loaded %s | %s | %s rows", source, path.name, len(raw))

if not frames:
    raise RuntimeError("No source files loaded")
df_raw = pd.concat(frames, ignore_index=True)
ingestion_report = pd.DataFrame(ingest_log)
display(ingestion_report)
print("Combined raw records:", f"{len(df_raw):,}")


In [ ]:
# Clean and deduplicate within each source (not across independent sources).
df = df_raw.copy()
df["review"] = clean_text(df["review"])
df["airline"] = clean_text(df["airline"])

# Stable identity: source review ID if present, otherwise hash of source/airline/date/text.
# An ID is namespaced by airline because some scrapers use local numeric IDs.
def stable_record_key(row):
    """Return or process stable record key."""
    parts = [str(row["source"]), str(row["airline"])]
    if pd.notna(row["review_id"]):
        parts += ["id", str(row["review_id"])]
    else:
        date = row["review_date"].strftime("%Y-%m-%d") if pd.notna(row["review_date"]) else ""
        parts += ["content", date, str(row["review"]).casefold()]
    return hashlib.sha256("\x1f".join(parts).encode("utf-8")).hexdigest()

df["record_key"] = df.apply(stable_record_key, axis=1)
initial_count = len(df)
df = df.drop_duplicates(subset=["record_key"], keep="first").reset_index(drop=True)
print(f"Removed {initial_count-len(df):,} duplicate source records")

# Keep empty-text rows in df for auditing, but exclude them from NLP summaries.
df["has_review"] = df["review"].notna()
print("Non-empty reviews:", f"{df['has_review'].sum():,}")
print("Empty reviews:", f"{(~df['has_review']).sum():,}")

# Ratings from different sources may be on different scales: retain raw values.
# Do not pool/average ratings across sources until source-specific scales are verified.

def review_summary(data):
    """Return or process review summary."""
    subset = data.loc[data["has_review"]]
    if subset.empty:
        return pd.DataFrame()
    return pd.crosstab(subset["airline"], subset["source"], margins=True, margins_name="Total")

table_all = review_summary(df)
table_2024 = review_summary(df.loc[df["review_date"] >= pd.Timestamp("2024-01-01")])
table_2026 = review_summary(df.loc[df["review_date"] >= pd.Timestamp("2026-01-01")])

for label, table in [("All reviews", table_all), ("2024 onwards", table_2024), ("2026 onwards", table_2026)]:
    print("\n", label)
    display(table)

with pd.ExcelWriter(OUTPUT_DIR / "airline_review_summary.xlsx", engine="openpyxl") as writer:
    table_all.to_excel(writer, sheet_name="All Reviews")
    table_2024.to_excel(writer, sheet_name="From 2024")
    table_2026.to_excel(writer, sheet_name="From 2026")
    ingestion_report.to_excel(writer, sheet_name="Ingestion Audit", index=False)

df.to_csv(OUTPUT_DIR / "airline_reviews_cleaned.csv", index=False, encoding="utf-8-sig")


In [ ]:
from lingua import LanguageDetectorBuilder
from tqdm.auto import tqdm

tqdm.pandas()

detector = LanguageDetectorBuilder.from_all_languages().build()

def detect_language_with_confidence(value):
    """Return or process detect language with confidence."""
    if pd.isna(value) or not str(value).strip():
        return pd.Series({"original_language": pd.NA, "language_confidence": np.nan})
    values = detector.compute_language_confidence_values(str(value))
    if not values:
        return pd.Series({"original_language": "Unknown", "language_confidence": 0.0})
    best = values[0]
    return pd.Series({"original_language": best.language.name.title(),
                      "language_confidence": float(best.value)})

language_results = df["review"].progress_apply(detect_language_with_confidence)
df = pd.concat([df, language_results], axis=1)
df["language"] = df["original_language"]  # Backwards compatibility

language_summary = df["original_language"].value_counts(dropna=False).rename_axis("language").reset_index(name="reviews")
language_summary["percentage"] = (100 * language_summary["reviews"] / len(df)).round(2)
display(language_summary)


In [ ]:
# Restore translations by stable record key rather than row order.
# This works even when new source files or new reviews are added.
CHECKPOINT_FILE = OUTPUT_DIR / "translation_checkpoint.pkl"
TRANSLATION_CSV = OUTPUT_DIR / "airline_reviews_translated.csv"

# Hash review text as well: a changed review must not reuse a stale translation.
def review_hash(value):
    """Return or process review hash."""
    return hashlib.sha256(str(value).encode("utf-8")).hexdigest() if pd.notna(value) else ""

df_final = df.copy()
df_final["review_hash"] = df_final["review"].map(review_hash)
df_final["review_en"] = df_final["review"].copy()
df_final["translated"] = False
df_final["translation_status"] = "pending"
df_final.loc[~df_final["has_review"], ["review_en", "translation_status"]] = [pd.NA, "empty"]
df_final.loc[df_final["original_language"].eq("English") & df_final["has_review"], "translation_status"] = "original_english"

if CHECKPOINT_FILE.exists():
    previous = pd.read_pickle(CHECKPOINT_FILE)
    required = {"record_key", "review_hash", "review_en", "translation_status"}
    if required.issubset(previous.columns):
        old = previous.drop_duplicates("record_key").set_index("record_key")
        valid = df_final["record_key"].isin(old.index)
        aligned = old.reindex(df_final["record_key"])
        matching = valid.to_numpy() & (df_final["review_hash"].to_numpy() == aligned["review_hash"].to_numpy())
        successful = aligned["translation_status"].isin(["completed", "original_english"]).to_numpy()
        reuse = matching & successful
        df_final.loc[reuse, "review_en"] = aligned.loc[reuse, "review_en"].to_numpy()
        df_final.loc[reuse, "translation_status"] = aligned.loc[reuse, "translation_status"].to_numpy()
        df_final.loc[reuse, "translated"] = df_final.loc[reuse, "translation_status"].eq("completed")
        print(f"Reused {reuse.sum():,} previously completed records")
    else:
        logger.warning("Checkpoint format incompatible; starting a new translation state")

def save_progress():
    """Return or process save progress."""
    df_final.to_pickle(CHECKPOINT_FILE)
    df_final.to_csv(TRANSLATION_CSV, index=False, encoding="utf-8-sig")

save_progress()
print(df_final["translation_status"].value_counts(dropna=False))


In [ ]:
# Review approximate API workload BEFORE enabling translation.
need_translation = df_final["translation_status"].isin(["pending", "error"])
pending = df_final.loc[need_translation]
word_count = pending["review"].fillna("").str.split().str.len().sum()
print(f"Pending translations: {len(pending):,}")
print(f"Approximate words to translate: {word_count:,.0f}")
print("API billing depends on actual model pricing and token usage; no hardcoded price is assumed.")


In [ ]:
if TRANSLATE_REVIEWS and need_translation.any():
    from openai import OpenAI
    api_key = os.getenv("OPENAI_API_KEY") or getpass("Enter OpenAI API key (not saved): ")
    client = OpenAI(api_key=api_key, timeout=90.0, max_retries=2)

    def translate_review(text, language):
        """Return or process translate review."""
        response = client.responses.create(
            model=TRANSLATION_MODEL,
            instructions=(
                "Translate this passenger review into natural English. "
                f"Detected original language: {language}. "
                "Preserve meaning, sentiment, criticism, sarcasm, numbers, airline names, "
                "airport codes and flight numbers. Do not summarize or add information. "
                "If already English, return unchanged. Return only translated review text."
            ),
            input=str(text),
            max_output_tokens=4000,
        )
        translated = (response.output_text or "").strip()
        if not translated:
            raise ValueError("Translation response was empty")
        usage = response.usage
        return translated, int(getattr(usage, "input_tokens", 0) or 0), int(getattr(usage, "output_tokens", 0) or 0)

    translation_cache = {
        str(row.review): str(row.review_en)
        for row in df_final.itertuples()
        if row.translation_status == "completed" and pd.notna(row.review_en)
    }
    tokens_in = tokens_out = translated_count = errors = 0
    indices = df_final.index[df_final["translation_status"].isin(["pending", "error"])]

    for counter, idx in enumerate(tqdm(indices, desc="Translating reviews"), 1):
        original = str(df_final.at[idx, "review"])
        try:
            if original in translation_cache:
                translated, input_tokens, output_tokens = translation_cache[original], 0, 0
            else:
                for attempt in range(MAX_TRANSLATION_ATTEMPTS):
                    try:
                        translated, input_tokens, output_tokens = translate_review(
                            original, df_final.at[idx, "original_language"]
                        )
                        break
                    except Exception:
                        if attempt + 1 == MAX_TRANSLATION_ATTEMPTS:
                            raise
                        time.sleep(min(2 ** attempt * 2, 30))
                translation_cache[original] = translated
                time.sleep(SLEEP_BETWEEN_CALLS)
            df_final.at[idx, "review_en"] = translated
            df_final.at[idx, "translated"] = True
            df_final.at[idx, "translation_status"] = "completed"
            tokens_in += input_tokens
            tokens_out += output_tokens
            translated_count += 1
        except Exception as exc:
            df_final.at[idx, "translation_status"] = "error"
            errors += 1
            logger.error("Translation failed for record %s: %s", df_final.at[idx, "record_key"][:12], exc)
        if counter % SAVE_EVERY == 0:
            save_progress()
            logger.info("Processed %s/%s | translated=%s | errors=%s", counter, len(indices), translated_count, errors)

    save_progress()
    print(f"Session: {translated_count:,} translated, {errors:,} errors")
    print(f"API tokens: {tokens_in:,} input / {tokens_out:,} output")
else:
    print("Translation skipped: disabled or no pending reviews")

save_progress()
print("Final translation statuses:")
display(df_final["translation_status"].value_counts(dropna=False))


## Optional long-haul classification

The original notebook included a route/duration classifier. This stage is intentionally separate from translation. It produces a binary `long_haul` column where **1 = evidence of a flight lasting at least six hours**, and **0 = not identified as long-haul**. The zero class includes unknown journeys; use `haul_type` to distinguish them. Airport-based estimates are approximate and should not be treated as actual flight durations.


In [ ]:
RUN_LONG_HAUL_CLASSIFICATION = False

if RUN_LONG_HAUL_CLASSIFICATION:
    import airportsdata
    from math import asin, cos, radians, sin, sqrt
    airports = airportsdata.load("IATA")

    def estimated_hours(origin, destination):
        """Return or process estimated hours."""
        if pd.isna(origin) or pd.isna(destination):
            return np.nan
        a, b = str(origin).upper().strip(), str(destination).upper().strip()
        if a not in airports or b not in airports or a == b:
            return np.nan
        x, y = airports[a], airports[b]
        lat1, lon1, lat2, lon2 = map(radians, (x["lat"], x["lon"], y["lat"], y["lon"]))
        h = sin((lat2-lat1)/2)**2 + cos(lat1)*cos(lat2)*sin((lon2-lon1)/2)**2
        km = 6371 * 2 * asin(min(1, sqrt(h)))
        return km / 800 + 0.5

    def classify_long_haul(row):
        """Return or process classify long haul."""
        review = str(row["review_en"] if pd.notna(row["review_en"]) else row["review"] or "")
        explicit = re.search(r"\b(\d+(?:[.,]\d+)?)\s*(?:hours?|hrs?)\s*(?:flight|journey)\b", review, re.I)
        if explicit:
            duration = float(explicit.group(1).replace(",", "."))
            return pd.Series({"haul_type": "Long-haul" if duration >= 6 else "Short/medium-haul", "haul_method": "explicit_duration"})
        origin, destination = row["origin"], row["destination"]
        if pd.notna(row["route"]):
            m = re.search(r"\b([A-Z]{3})\s*[-–—→/]\s*([A-Z]{3})\b", str(row["route"]).upper())
            if m and (pd.isna(origin) or pd.isna(destination)):
                origin, destination = m.groups()
        hours = estimated_hours(origin, destination)
        if pd.notna(hours):
            if hours >= 6.5:
                return pd.Series({"haul_type": "Long-haul", "haul_method": "estimated_route"})
            if hours < 5.5:
                return pd.Series({"haul_type": "Short/medium-haul", "haul_method": "estimated_route"})
            return pd.Series({"haul_type": "Unknown", "haul_method": "borderline_estimate"})
        if re.search(r"\blong[- ]haul\b", review, re.I):
            return pd.Series({"haul_type": "Long-haul", "haul_method": "text_mention"})
        if re.search(r"\bshort[- ]haul\b", review, re.I):
            return pd.Series({"haul_type": "Short/medium-haul", "haul_method": "text_mention"})
        return pd.Series({"haul_type": "Unknown", "haul_method": "insufficient_information"})

    df_final[["haul_type", "haul_method"]] = df_final.progress_apply(classify_long_haul, axis=1)
    df_final["long_haul"] = df_final["haul_type"].eq("Long-haul").astype(int)
    display(pd.crosstab(df_final["airline"], df_final["haul_type"], margins=True))


In [ ]:
# Final exports and validation.
FINAL_CSV = OUTPUT_DIR / "df_final.csv"
FINAL_PARQUET = OUTPUT_DIR / "df_final.parquet"

df_final.to_csv(FINAL_CSV, index=False, encoding="utf-8-sig")
try:
    df_final.to_parquet(FINAL_PARQUET, index=False)
except ImportError:
    logger.info("Parquet skipped; install pyarrow to enable it")

print("Rows:", f"{len(df_final):,}")
print("Non-empty review text:", f"{df_final['has_review'].sum():,}")
print("Untranslated non-English reviews:", f"{df_final['translation_status'].isin(['pending', 'error']).sum():,}")
print("Unknown dates:", f"{df_final['review_date'].isna().sum():,}")
print("Unknown airlines:", f"{df_final['airline'].isna().sum():,}")
print("Saved:", FINAL_CSV)
display(df_final[["airline", "source", "review_date", "rating", "review",
                  "original_language", "review_en", "translation_status"]].head(12))
